In [3]:
# Cell 1: Imports & Configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import os
import random
import json

# --- Sonar Array Configuration ---
C = 1500.0       # Speed of sound in water (m/s)
NUM_SENSORS = 100 # Number of hydrophones (M) - slightly larger array for better resolution
FS = 8000        # Sampling rate for simulation.
                 # 8kHz is enough to capture engine noise (up to 4kHz) and saves RAM.

# Design Frequency (for sensor spacing)
F_DESIGN = 2000  # Frequency for lambda/2 spacing calculation
LAMBDA = C / F_DESIGN
D_SPACING = LAMBDA / 2

print(f"--- Simulation Setup ---")
print(f"Array: {NUM_SENSORS} sensors, Spacing: {D_SPACING:.3f} m")
print(f"Sampling Rate: {FS} Hz")

--- Simulation Setup ---
Array: 100 sensors, Spacing: 0.375 m
Sampling Rate: 8000 Hz


In [4]:
# Cell 2: Load Dataset Metadata
JSON_FILE = 'vessels.json'  # Ensure this file is in your notebook directory
DATA_ROOT = r'C:\Users\akiva\OneDrive\Desktop\Final Project' # <--- UPDATE THIS PATH to your real data folder!

def load_metadata(json_path):
    if not os.path.exists(json_path):
        print(f"Error: {json_path} not found.")
        return pd.DataFrame()

    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    # Clean paths to match your Operating System (fixing the backslashes from the JSON)
    # The JSON uses backslashes like "\data\sounds..."
    df['sound_path'] = df['sound'].apply(lambda x: x.replace('\\', os.sep).lstrip(os.sep) if x else None)

    # Filter out entries that have no sound file
    df = df.dropna(subset=['sound_path'])

    print(f"Loaded {len(df)} valid records.")
    print("Categories found:", df['category'].unique()) # e.g., Yacht, Motorboat, etc.
    return df

df = load_metadata(JSON_FILE)
df.head(3)

Loaded 1149 valid records.
Categories found: ['Yacht' 'Motorboat' 'Sailboat' 'Tour boat' 'Auxiliary ship' 'Ferry'
 'Fishing Vessel' 'Cargo Ship' 'Tour Boat - Diving boat']


,id,name,category,subcategory,imoNumber,speed,length,pressure,latitude,longitude,thirdOctaveData,time,image,sound,video,sound_path
0,1,,Yacht,Monohull Cruiser,0,4.3,20.50,171.12,None,None,"{'40': 138.26, '50': 146.539, '63': 158.716, '...",2023-08-02T08:59:55.000Z,\data\sounds-and-images\Yachts\Yacht_02.08.23_...,\data\sounds-and-images\Yachts\Yacht_02.08.23_...,\data\videos\20230802_105955.mp4,data\sounds-and-images\Yachts\Yacht_02.08.23_1...
1,2,,Motorboat,Motor boat-Outboard engine,0,5.3,9.77,174.10,None,None,"{'40': 162.035, '50': 155.052, '63': 149.17, '...",2023-08-02T09:05:47.000Z,\data\sounds-and-images\Motor Boats\Motorboat_...,\data\sounds-and-images\Motor Boats\Motorboat_...,\data\videos\20230802_110547.mp4,data\sounds-and-images\Motor Boats\Motorboat_0...
2,3,,Yacht,Monohull Cruiser,0,4.7,18.80,174.99,None,None,"{'40': 159.405, '50': 159.049, '63': 150.191, ...",2023-08-02T09:15:40.000Z,\data\sounds-and-images\Yachts\Yacht_02.08.23_...,\data\sounds-and-images\Yachts\Yacht_02.08.23_...,\data\videos\20230802_111540.mp4,data\sounds-and-images\Yachts\Yacht_02.08.23_1...


In [5]:
import os
import json
import requests
import time
from urllib.parse import urljoin

# --- Configuration ---
BASE_URL = "https://hearmyship.fer.hr"

# 1. תיקון הנתיב: מצביעים לקובץ הספציפי, לא לתיקייה
# השתמשתי ב-os.path.join כדי למנוע בעיות עם לוכסנים
BASE_DIR = r"C:\Users\akiva\OneDrive\שולחן העבודה\Final Project"
JSON_FILE = os.path.join(BASE_DIR, "data", "vessels.json")
OUTPUT_DIR = os.path.join(BASE_DIR, "data")

def download_all_audio():
    # בדיקה שהקובץ קיים
    if not os.path.exists(JSON_FILE):
        print(f"❌ Error: Could not find JSON file at: {JSON_FILE}")
        return

    print(f"Reading map from: {JSON_FILE}")

    with open(JSON_FILE, 'r', encoding='utf-8') as f:
        records = json.load(f)

    print(f"Found {len(records)} records. Starting download...")

    success_count = 0

    for i, record in enumerate(records):
        # שליפת הנתיב מה-JSON (למשל: \data\sounds-and-images\...)
        relative_path = record.get('sound')

        if not relative_path:
            continue

        # הכנת הקישור להורדה (Web URL)
        # הופכים לוכסנים של ווינדוס (\) ללוכסנים של אינטרנט (/)
        web_path = relative_path.replace('\\', '/').lstrip('/')
        download_url = urljoin(BASE_URL, web_path)

        # הכנת הנתיב לשמירה במחשב
        # אנו מנקים את המילה "data" מהתחלה כדי לא ליצור כפילות (data/data/...)
        clean_local_path = relative_path.replace('\\data\\', '').replace('/data/', '').lstrip(os.sep)
        save_path = os.path.join(OUTPUT_DIR, clean_local_path)

        # יצירת התיקיות אם הן לא קיימות (למשל sounds-and-images/Yachts)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # הורדה רק אם הקובץ עדיין לא קיים
        if not os.path.exists(save_path):
            print(f"[{i+1}/{len(records)}] Downloading: {clean_local_path}...")
            try:
                response = requests.get(download_url, stream=True)
                if response.status_code == 200:
                    with open(save_path, 'wb') as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    success_count += 1
                    # הפסקה קטנה כדי לא להעמיס על השרת שלהם
                    time.sleep(0.1)
                else:
                    print(f"   ❌ Failed (Status {response.status_code})")
            except Exception as e:
                print(f"   ❌ Error: {e}")
        else:
            # אם הקובץ קיים, מדלגים עליו (חוסך זמן אם מריצים שוב)
            # print(f"[{i+1}/{len(records)}]Exists: {clean_local_path}")
            pass

    print(f"\n✨ Finished! Downloaded {success_count} new files.")

if __name__ == "__main__":
    download_all_audio()

Reading map from: C:\Users\akiva\OneDrive\שולחן העבודה\Final Project\data\vessels.json
Found 1149 records. Starting download...
[249/1149] Downloading: sounds-and-images\Tour Boat\TourBoat_A_Blue_White_Orange_Bouys\Tourboat_1_17.08.23_150933\Tourboat_%231_17.08.23_150933_20secCPA.wav...
   ❌ Failed (Status 404)
[287/1149] Downloading: sounds-and-images\Tour Boat\TourBoat_B_Dark_Green_White_Bouys\TourBoat_2_18.08.23_124348\TourBoat_%232_18.08.23_124348_20secCPA.wav...
   ❌ Failed (Status 404)
[733/1149] Downloading: sounds-and-images\Tour Boat\TourBoat_A_Blue_White_Orange_Bouys\Tourboat_1_28.08.23_183246\Tourboat_%231_28.08.23_183246_20secCPA.wav...
   ❌ Failed (Status 404)

✨ Finished! Downloaded 0 new files.


In [6]:
import os
import json

# --- הגדרות נתיב ---
# ודא שזה הנתיב לתיקייה הראשית של הפרויקט שלך
BASE_DIR = r"C:\Users\akiva\OneDrive\שולחן העבודה\Final Project"
JSON_PATH = os.path.join(BASE_DIR, "data", "vessels.json")
DATA_DIR = os.path.join(BASE_DIR, "data")

def check_downloads():
    if not os.path.exists(JSON_PATH):
        print(f"❌ Error: Cannot find vessels.json at: {JSON_PATH}")
        return

    print("📖 Reading manifest...")
    with open(JSON_PATH, 'r', encoding='utf-8') as f:
        records = json.load(f)

    total_files = 0
    found_files = 0
    missing_files = []

    print(f"🔍 Checking {len(records)} records...")

    for record in records:
        # נתיב מקורי מה-JSON
        relative_path = record.get('sound')

        if not relative_path:
            continue

        total_files += 1

        # המרה לנתיב מקומי נקי
        # מוחקים את 'data' מההתחלה כדי להתאים למבנה התיקיות שלך
        clean_path = relative_path.replace('\\data\\', '').replace('/data/', '').lstrip(os.sep)
        full_local_path = os.path.join(DATA_DIR, clean_path)

        # בדיקה פיזית בדיסק
        if os.path.exists(full_local_path):
            found_files += 1
            # בדיקת בונוס: האם הקובץ לא ריק (גודל 0)?
            if os.path.getsize(full_local_path) == 0:
                print(f"⚠️ Warning: File exists but is empty (0 bytes): {clean_path}")
        else:
            missing_files.append(clean_path)

    # --- דוח סיכום ---
    print("\n" + "="*30)
    print("       📊 Download Report       ")
    print("="*30)
    print(f"Total Audio Records:  {total_files}")
    print(f"✅ Successfully Found: {found_files}")
    print(f"❌ Missing Files:      {len(missing_files)}")
    print("-" * 30)

    if len(missing_files) == 0:
        print("🎉 PERFECT! All files are present and accounted for.")
        print("You are ready to start the Beamforming simulation.")
    else:
        print("⚠️ Some files are missing. First 5 missing files:")
        for f in missing_files[:5]:
            print(f" - {f}")
        print(f"... and {len(missing_files) - 5} more.")
        print("Tip: Run the downloader script again to fetch the missing ones.")

if __name__ == "__main__":
    check_downloads()

📖 Reading manifest...
🔍 Checking 1149 records...

       📊 Download Report       
Total Audio Records:  1149
✅ Successfully Found: 1146
❌ Missing Files:      3
------------------------------
⚠️ Some files are missing. First 5 missing files:
 - sounds-and-images\Tour Boat\TourBoat_A_Blue_White_Orange_Bouys\Tourboat_1_17.08.23_150933\Tourboat_%231_17.08.23_150933_20secCPA.wav
 - sounds-and-images\Tour Boat\TourBoat_B_Dark_Green_White_Bouys\TourBoat_2_18.08.23_124348\TourBoat_%232_18.08.23_124348_20secCPA.wav
 - sounds-and-images\Tour Boat\TourBoat_A_Blue_White_Orange_Bouys\Tourboat_1_28.08.23_183246\Tourboat_%231_28.08.23_183246_20secCPA.wav
... and -2 more.
Tip: Run the downloader script again to fetch the missing ones.
